# 01 · GWAS 基础概念 GWAS Introduction

> **能力标签**：GB1（GWAS 基础概念 / GWAS Fundamentals）、GB2（关联测试与表型 / Association Testing）

## 目标 Objectives

完成本课后，你应该能够：

1. 解释 **SNP**（单核苷酸多态性 single-nucleotide polymorphism）、**等位基因（allele）**、**剂量（dosage）** 的含义。
2. 计算 **次要等位基因频率**（MAF, minor allele frequency）与 **次要等位基因计数**（MAC, minor allele count）。
3. 区分 **关联 vs 因果**（association vs causation）——GWAS 找的是统计关联，不是机制。
4. 说明 **病例-对照研究**（case-control study）与 **数量性状**（quantitative trait）在 GWAS 中的区别。
5. 用合成基因型矩阵验证自己对 MAF/MAC 的理解，为 b7-maf 作业做好准备。

> 对应能力：**GB1** · **GB2**


## 概念讲解 Concepts

### 1 · SNP 与等位基因 SNP & Alleles

**SNP**（发音"snip"）是基因组中最常见的变异类型：在特定位点，不同个体可能携带不同的碱基（nucleotide）。
典型双等位 SNP（biallelic SNP）只有两种碱基，通常记为**参考等位基因**（reference allele, REF）和**替代等位基因**（alternate allele, ALT）。

人类是二倍体（diploid），每个常染色体位点有两个拷贝（copy/allele）。因此每个体在某 SNP 的**基因型**（genotype）有三种：

| 基因型 | 含义 | 剂量（dosage）|
|--------|------|--------------|
| REF/REF | 纯合参考（homozygous reference） | 0 |
| REF/ALT | 杂合（heterozygous） | 1 |
| ALT/ALT | 纯合替代（homozygous alternate） | 2 |

**剂量（dosage）** = 替代等位基因的拷贝数，范围 {0, 1, 2}。
在矩阵 $G$ 中，行是样本（samples），列是 SNP（variants）：$G \in \{0,1,2\}^{n \times m}$。

---

### 2 · 次要等位基因频率与计数 MAF & MAC

设共 $n$ 个样本，SNP $j$ 的替代等位基因计数（alternate allele count, AAC）为

$$\text{AAC}_j = \sum_{i=1}^{n} G_{ij}$$

总等位基因数 $= 2n$（每人两个拷贝），替代等位基因频率（AAF）：

$$\text{AAF}_j = \frac{\text{AAC}_j}{2n}$$

**MAF（minor allele frequency）** 取 AAF 与其补数中的较小值：

$$\text{MAF}_j = \min(\text{AAF}_j,\ 1 - \text{AAF}_j)$$

**MAC（minor allele count）** 类似，取两侧计数的较小值：

$$\text{MAC}_j = \min(\text{AAC}_j,\ 2n - \text{AAC}_j)$$

实践中常过滤掉 MAF < 0.01（稀有变异）或 MAC < 20 的 SNP，以保证统计检验的功效（power）。

---

### 3 · GWAS 是什么？关联 vs 因果 What is GWAS?

**GWAS（全基因组关联研究，genome-wide association study）**：
在整个基因组范围内，逐个检验数十万至数百万个 SNP 是否与感兴趣的表型（phenotype）存在**统计关联**（statistical association）。

> **关联不等于因果（association != causation）**
>
> GWAS 发现的关联可能源于：
> - 真实的直接效应（direct causal effect）
> - 连锁不平衡（linkage disequilibrium, LD）：因果变异附近的搭便车 SNP
> - 群体分层（population stratification）：混杂因子

---

### 4 · 表型类型 Phenotype Types

| 类型 | 描述 | 示例 |
|------|------|------|
| **病例-对照（case-control）** | 二元（binary）结局：有病/无病 | 2型糖尿病、精神分裂症 |
| **数量性状（quantitative trait）** | 连续（continuous）结局 | 身高、BMI、血脂水平 |

病例-对照可用逻辑回归（logistic regression）或线性概率模型；
数量性状通常用线性回归（linear regression）——本周的核心。


## 示例 Worked Example

用 NumPy 生成合成基因型矩阵，计算 MAF 与 MAC，并可视化频率分布。

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tempfile, os

rng = np.random.default_rng(42)

# 合成基因型矩阵 G: (n_samples=200, n_snps=500)
# 用二项分布模拟剂量：每个 SNP 有自己的替代等位基因频率
n, m = 200, 500
true_aaf = rng.uniform(0.05, 0.95, size=m)
G = rng.binomial(n=2, p=true_aaf, size=(n, m)).astype(float)

print(f"G.shape = {G.shape}  (samples x SNPs)")
print("G 前 3 行、前 5 列：")
print(G[:3, :5])


In [ ]:
# 计算 MAF
def maf(G):
    """MAF for each SNP: min(freq, 1-freq).  G: (n, m) dosage matrix."""
    G = np.asarray(G, dtype=float)
    p = G.mean(axis=0) / 2.0
    return np.minimum(p, 1.0 - p)

# 计算 MAC
def mac(G):
    """MAC for each SNP: min(AC, 2n-AC).  G: (n, m) dosage matrix."""
    G = np.asarray(G, dtype=float)
    ac = G.sum(axis=0)
    n_total = 2 * G.shape[0]
    return np.minimum(ac, n_total - ac)

maf_vals = maf(G)
mac_vals = mac(G)

print(f"MAF: min={maf_vals.min():.4f}  max={maf_vals.max():.4f}  mean={maf_vals.mean():.4f}")
print(f"MAC: min={mac_vals.min():.0f}  max={mac_vals.max():.0f}  mean={mac_vals.mean():.1f}")
mac_from_maf = maf_vals * 2 * n
print(f"max |MAC - MAF*2n| = {np.max(np.abs(mac_vals - mac_from_maf)):.2f}  (应接近 0)")


In [ ]:
# 可视化 MAF 分布
tmpdir = tempfile.mkdtemp()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(maf_vals, bins=30, color="steelblue", edgecolor="white", linewidth=0.5)
axes[0].axvline(0.01, color="red", linestyle="--", label="MAF=0.01 (常见过滤阈值)")
axes[0].set_xlabel("MAF")
axes[0].set_ylabel("SNP 数量")
axes[0].set_title("MAF 分布 Distribution")
axes[0].legend()

axes[1].scatter(maf_vals, mac_vals, alpha=0.3, s=8, color="steelblue")
axes[1].set_xlabel("MAF")
axes[1].set_ylabel("MAC")
axes[1].set_title("MAC vs MAF (应严格正比)")

fig.tight_layout()
out = os.path.join(tmpdir, "maf_mac.png")
fig.savefig(out, dpi=100)
plt.close(fig)
print(f"图片已保存：{out}")

keep = maf_vals >= 0.01
print(f"MAF >= 0.01 的 SNP：{keep.sum()} / {m}  ({keep.mean()*100:.1f}%)")


In [ ]:
# 数值验证：与真实频率对比
true_maf = np.minimum(true_aaf, 1 - true_aaf)
error = np.abs(maf_vals - true_maf)
print(f"MAF 估计误差：mean={error.mean():.4f}  max={error.max():.4f}")
print("(样本量 n=200 下的抽样误差，符合预期)")

j = 0
g0 = G[:, j]
aac0 = g0.sum()
n_alleles = 2 * n
aaf0 = aac0 / n_alleles
maf0 = min(aaf0, 1 - aaf0)
mac0 = min(aac0, n_alleles - aac0)
print(f"SNP 0 手动计算：AAC={aac0:.0f}, AAF={aaf0:.4f}, MAF={maf0:.4f}, MAC={mac0:.0f}")
print(f"函数计算：       MAF={maf_vals[0]:.4f}, MAC={mac_vals[0]:.0f}  (一致)")


## 动手 Exercise

以下合成基因型矩阵 `G_ex` 是一个小型示例（50 样本 × 10 SNP）。请你：

1. **手动计算** SNP 0 的 AAF、MAF 和 MAC（不调用上面定义的函数，用 numpy 基本操作）。
2. **调用** `maf(G_ex)` 和 `mac(G_ex)`，对比结果是否一致。
3. **计数** MAF < 0.05 的 SNP 有几个？将其从矩阵中移除后，输出过滤后矩阵的 shape。

```python
rng_ex = np.random.default_rng(7)
true_aaf_ex = rng_ex.uniform(0.0, 0.5, size=10)
G_ex = rng_ex.binomial(2, true_aaf_ex, size=(50, 10)).astype(float)

# 你的代码写在这里
```

> **提示**：注意 MAC 与 MAF 在边界情况（全部为 REF/REF 时）的行为。


## 延伸阅读 Further Reading

1. **Tam et al. (2019)**. "Benefits and limitations of genome-wide association studies." *Nature Reviews Genetics.*
2. **Visscher et al. (2017)**. "10 years of GWAS discovery: biology, function, and translation." *American Journal of Human Genetics.*
3. GWAS 概念综述（中文）：可搜索"Mendelian randomization 因果推断"了解遗传学因果推断方法。
4. `01-association-testing/README.md`——本周课程结构与作业说明（离线可读）。


## 关联练习 Related Assignment

完成 **b7-maf** 编程作业：在 `progress/<你>/work/b7-maf/solution.py` 中实现
`maf(G)` 和 `mac(G)` 函数。

评测命令：

```bash
python check.py 01-maf
```

> 作业函数签名与本课示例完全一致，直接参照上方代码实现即可。
